chuyển csv thành json

In [1]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv(r"C:\Users\Admin\Downloads\quanbucacchogmail_com\product_nutrition.csv")

# Xuất JSON
df.to_json(
    r"C:\Users\Admin\Downloads\quanbucacchogmail_com\product_nutrition.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("Đã chuyển xong")

Đã chuyển xong


In [2]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv(r"C:\Users\Admin\Downloads\quanbucacchogmail_com\products.csv")

# Xuất JSON
df.to_json(
    r"C:\Users\Admin\Downloads\quanbucacchogmail_com\products.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("Đã chuyển xong")

Đã chuyển xong


lưu tên sản phẩm vô file nào đó

In [4]:
import json

# Đọc file JSON
with open(r"C:\Users\Admin\Downloads\quanbucacchogmail_com\product_nutrition.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Lưu tên sản phẩm
with open(r"C:\Users\Admin\Downloads\quanbucacchogmail_com\product_names.txt", "w", encoding="utf-8") as f:
    for item in data:
        f.write(item["product_name"] + "\n")

print("Đã lưu xong")

Đã lưu xong


code crawl ảnh sản phẩm

In [8]:
options.add_argument("--disable-blink-features=AutomationControlled")

In [10]:
driver.execute_script("""
Object.defineProperty(navigator, 'webdriver', {
    get: () => undefined
})
""")

In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, random, re

INPUT_FILE  = r"C:\Users\Admin\Downloads\quanbucacchogmail_com\product_names.txt"
OUTPUT_FILE = r"C:\Users\Admin\Downloads\quanbucacchogmail_com\url_pic2.txt"

def random_delay(a=1.5, b=3.5):
    time.sleep(random.uniform(a, b))

def build_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    # options.binary_location = r"C:\Program Files\CocCoc\Browser\Application\browser.exe"

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
            window.chrome = { runtime: {} };
            Object.defineProperty(navigator, 'languages', { get: () => ['vi-VN', 'vi', 'en-US'] });
            Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3] });
        """
    })
    return driver

def accept_cookies(driver):
    try:
        btn = driver.find_element(By.XPATH,
            '//button[.//span[contains(text(),"Accept") or contains(text(),"Đồng ý") or contains(text(),"Agree")]]')
        btn.click()
        random_delay(1, 2)
    except:
        pass

def get_image_url(driver, product):
    wait = WebDriverWait(driver, 10)
    query = product.replace(" ", "+")
    driver.get(f"https://www.google.com/search?q={query}&tbm=isch&hl=vi")
    random_delay(2, 3)
    accept_cookies(driver)

    # ── Cách 1: lấy từ thẻ <a> chứa imgurl (không cần click) ──
    try:
        anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='imgurl=']")
        for a in anchors:
            href = a.get_attribute("href") or ""
            match = re.search(r"imgurl=([^&]+)", href)
            if match:
                from urllib.parse import unquote
                url = unquote(match.group(1))
                if url.startswith("http"):
                    print(f"  [method=href] {url[:80]}")
                    return url
    except Exception as e:
        print(f"  [href method fail] {e}")

    # ── Cách 2: click ảnh đầu tiên → lấy URL trong panel bên phải ──
    try:
        # Tìm thumbnail đầu tiên hợp lệ
        thumbnails = wait.until(EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "div[data-ved] img, img.YQ4gaf, img[jsname='Q4LuWd']")
        ))
        clicked = False
        for thumb in thumbnails[:5]:
            src = thumb.get_attribute("src") or ""
            if "data:image" in src or not src:
                continue  # bỏ qua placeholder base64
            try:
                driver.execute_script("arguments[0].click();", thumb)
                clicked = True
                break
            except:
                continue

        if not clicked:
            # fallback: click thumbnail đầu tiên bất kể src
            driver.execute_script("arguments[0].click();", thumbnails[0])

        random_delay(1.5, 2.5)

        # Panel bên phải hiện ra — tìm ảnh full size
        full_selectors = [
            "img.iPVvYb",           # full size image panel
            "img.r48jcc",
            "img[jsname='kn3ccd']",
            "div.tvh9oe img",
            "c-wiz img[src^='http']",
        ]
        for sel in full_selectors:
            try:
                imgs = driver.find_elements(By.CSS_SELECTOR, sel)
                for img in imgs:
                    src = img.get_attribute("src") or ""
                    if src.startswith("http") and "gstatic" not in src and "google" not in src:
                        print(f"  [method=click panel] {src[:80]}")
                        return src
            except:
                continue
    except Exception as e:
        print(f"  [click method fail] {e}")

    # ── Cách 3: parse HTML source tìm URL ảnh trong JSON ──
    try:
        page = driver.page_source
        # Google nhúng URL ảnh thật trong JSON của page
        matches = re.findall(r'"(https://[^"]+\.(?:jpg|jpeg|png|webp))"', page)
        for url in matches:
            if "encrypted" not in url and "gstatic" not in url:
                print(f"  [method=regex] {url[:80]}")
                return url
    except Exception as e:
        print(f"  [regex method fail] {e}")

    return None

# ═══════════════════════════════
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    product_names = [line.strip() for line in f if line.strip()]

driver = build_driver()
results = []
RESTART_EVERY = 15

try:
    for i, product in enumerate(product_names):
        if i > 0 and i % RESTART_EVERY == 0:
            driver.quit()
            random_delay(8, 15)
            driver = build_driver()

        try:
            image_url = get_image_url(driver, product)
            if image_url:
                results.append(f"{product} | {image_url}")
                print(f"[OK] {product}")
            else:
                results.append(f"{product} | NOT_FOUND")
                print(f"[FAIL] {product}")
        except Exception as e:
            results.append(f"{product} | ERROR: {e}")
            print(f"[ERROR] {product}: {e}")

        random_delay(3, 6)
finally:
    driver.quit()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(results) + "\n")

print("DONE")

  [method=regex] https://www.optimumnutritionsea.com/frontend/images/products/whey_bottele.png
[OK] Optimum Nutrition 100% Gold Standard Whey
  [method=regex] https://m.media-amazon.com/images/I/51GxLvRtlTL._AC_UF1000,1000_QL80_.jpg
[OK] True Nutrition rBGH/Soy Free Whey Protein
  [method=regex] https://cloudinary.images-iherb.com/image/upload/f_auto,q_auto:eco/images/rmt/rm
[OK] Premier Protein Chocolate Milkshake
  [method=regex] https://product.hstatic.net/1000185761/product/vani_31241296cdda473ba58130fa5f59
[OK] MyProtein Impact Whey Isolate
  [method=regex] https://d3gqasl9vmjfd8.cloudfront.net/705de54d-c31e-4100-baa8-847e31e95e80.png
[OK] Naked Egg White Protein Powder
  [method=regex] https://cdn11.bigcommerce.com/s-hr7ra7xc8x/images/stencil/original/attribute_rul
[OK] Orgain Plant-Based Protein Powder
  [method=regex] https://m.media-amazon.com/images/I/616uIisXDrL.jpg
[OK] Bloom Whey Protein Isolate
  [method=regex] https://cdn.thuoc.com.vn/p/now-foods-sports-whey-protein-isol